# Cox survival analysis

* `mydir`: your own data
* `ostime_column`: the survival time of the data; it does not have to be OST, it can also be DST, FST, etc.
* `os`: survival status; it does not have to be OS, it can also be DS, FS, etc.

In [ ]:
from lifelines import CoxPHFitter
import pandas as pd
from onekey_algo.custom.components.comp1 import normalize_df
from sklearn.model_selection import train_test_split
from onekey_algo import get_param_in_cwd
from onekey_algo.custom.components.comp1 import fillna
from lifelines.utils import concordance_index

event_col = get_param_in_cwd('event_col')
group_info = 'group'
task_type = 'Combined_'
duration_col= get_param_in_cwd('duration_col')
data = pd.read_csv('features/Clinical_features_mulsel.csv')
# data = normalize_df(data, not_norm='ID')
label_data = pd.read_csv(get_param_in_cwd('label_file'))
label_data['ID'] = label_data['ID'].map(lambda x: f"{x}.nii.gz" if not (f"{x}".endswith('.nii.gz') or  f"{x}".endswith('.nii')) else x)

# Clinical_features_mulsel.csv already contains the group, event, duration columns; merge only if missing
if 'group' not in data.columns or event_col not in data.columns or duration_col not in data.columns:
    data = pd.merge(data, label_data[['ID', event_col, duration_col, 'group']], on='ID', how='inner')
data

In [ ]:
def get_sig(prefix):
    results = [pd.read_csv(f'results/{prefix}_cox_predictions_{subset}.csv')for subset in get_param_in_cwd('subsets')]
    results = pd.concat(results, axis=0)
    return results

mn = {'Radiomics': f'Radiomics'}
for sig in mn:
    signature = get_sig(mn[sig])
    signature.columns = ['ID', sig, '-']
    data = pd.merge(data, signature, on='ID', how='inner')
data

In [ ]:
import numpy as np
import onekey_algo.custom.components as okcomp
from collections import OrderedDict

group_info = 'group'
train_data = data[(data[group_info] == 'train')]

subsets = get_param_in_cwd('subsets')
val_datasets = OrderedDict()
for subset in subsets:
    val_data = data[data[group_info] == subset]
    val_datasets[subset] = val_data
    val_data.to_csv(f'features/{task_type}{subset}_cox.csv', index=False)

print(', '.join([f"{subset} sample count: {d_.shape}" for subset, d_ in val_datasets.items()]))

## Cox overview

All the data necessary for Cox regression; there are 3 main quantities of interest
1. `Concordance`: c-index
2. `exp(coef)`: the HR for each feature, together with its corresponding 95% percentile.
3. `p`: indicates whether the feature is significant.

In [ ]:
from lifelines import CoxPHFitter

cph = CoxPHFitter(penalizer=0.3)
cph.fit(train_data[[c for c in train_data.columns if c not in ['ID', 'group'] and not c.startswith('-')]], 
        duration_col=duration_col, event_col=event_col)
cph.print_summary()

#### Output the HR of each feature

In [ ]:
cph.concordance_index_

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, train_data.shape[1]-4))
cph.plot(hazard_ratios=True)
plt.savefig(f'img/{task_type}_feature_pvalue.svg')
plt.show()

In [ ]:
from onekey_algo.custom.components import nomogram
from onekey_algo import get_param_in_cwd

cox_data = train_data.round(decimals=2)
sur_name = get_param_in_cwd('time_settings')
nomogram.nomogram(train_data, duration=duration_col, result=event_col, 
                  columns=[c for c in train_data.columns if c not in ['ID', 'group', duration_col, event_col] and not c.startswith('-')],
                  survs=list(sur_name.values()), surv_names=list(sur_name.keys()), with_r=False,
                  width=7000, height=5000, save_name=f'img/nomogram.png',
                  x_range='0.01,0.25,0.5,0.75,0.95')

# KM curves

Group by HR and compute KM and the log-rank test

In [ ]:
from lifelines import CoxPHFitter
from lifelines.statistics import logrank_test
from lifelines import KaplanMeierFitter
from lifelines.plotting import add_at_risk_counts

# Determine the optimal cutoff using the log-rank maximization method (computed on the training set)
def find_optimal_cutoff_logrank(risk_scores, durations, events, min_group_ratio=0.1):
    """
    Find the optimal cutoff using the log-rank statistic maximization method
    The principle is consistent with X-tile software, but without multiple-comparison correction
    """
    n = len(risk_scores)
    min_size = int(n * min_group_ratio)
    
    percentiles = np.linspace(np.percentile(risk_scores, 10), 
                              np.percentile(risk_scores, 90), 100)
    
    best_stat = -np.inf
    best_cutoff = np.median(risk_scores)
    
    for cutoff in percentiles:
        high_mask = risk_scores > cutoff
        low_mask = risk_scores <= cutoff
        
        if sum(high_mask) < min_size or sum(low_mask) < min_size:
            continue
        if events[high_mask].sum() < 3 or events[low_mask].sum() < 3:
            continue
        
        try:
            result = logrank_test(
                durations[high_mask], durations[low_mask],
                events[high_mask], events[low_mask]
            )
            if result.test_statistic > best_stat:
                best_stat = result.test_statistic
                best_cutoff = cutoff
        except:
            continue
    
    return best_cutoff

# Determine the optimal cutoff on the training set
train_risk = cph.predict_partial_hazard(train_data[[c for c in train_data.columns if c not in ['ID', 'group'] and not c.startswith('-')]])
optimal_cutoff = find_optimal_cutoff_logrank(
    train_risk.values.flatten(),
    train_data[duration_col].values,
    train_data[event_col].values
)
print(f"\nOptimal cutoff on the training set (log-rank maximization): {optimal_cutoff:.4f}")

c_index_list = []
thres = 1e-4
bst_split = {'train': 1.0, 'val':1, 'test': 1}
loc = {'train': 0.5, 'val':0.5, 'test': 0.5}
for subset, test_data in val_datasets.items():
    c_index = cph.score(test_data[[c for c in test_data.columns if c != 'ID']], scoring_method="concordance_index")
    y_pred = cph.predict_partial_hazard(test_data[[c for c in test_data.columns if c != 'ID']])
    cox_data = pd.concat([test_data, y_pred], axis=1)
    # Use the optimal cutoff determined on the training set
    cox_data['HR'] = cox_data[0] > optimal_cutoff

    dem = (cox_data["HR"] == True)
    results = logrank_test(cox_data[duration_col][dem], cox_data[duration_col][~dem], 
                           event_observed_A=cox_data[event_col][dem], event_observed_B=cox_data[event_col][~dem])
    p_value = f"={results.p_value:.3f}" if results.p_value > thres else f'<{thres}'
    plt.title(f"Cohort {subset} C-index:{c_index:.3f}")
    plt.ylabel('Probability')
    if sum(dem):
        kmf_high = KaplanMeierFitter()
        kmf_high.fit(cox_data[duration_col][dem], event_observed=cox_data[event_col][dem], label="High Risk")
        kmf_high.plot_survival_function(color='r')
    if sum(~dem):
        kmf_low = KaplanMeierFitter()
        kmf_low.fit(cox_data[duration_col][~dem], event_observed=cox_data[event_col][~dem], label="Low Risk")
        kmf_low.plot_survival_function(color='g')
    plt.text(0.5, loc[subset] if subset in loc else 0.2, f"P{p_value}")
    plt.xlabel('Time(months)')
    add_at_risk_counts(kmf_high, kmf_low, rows_to_show=['At risk'])
    plt.savefig(f'img/{task_type}KM_{subset}.svg', bbox_inches='tight')
    plt.show()

In [ ]:
import os
import numpy as np

def get_prediction(model: CoxPHFitter, data, ID=None, **kwargs):
    hr = model.predict_partial_hazard(data)
    expectation = model.predict_expectation(data)
    
    predictions = pd.concat([hr, expectation], axis=1)
    predictions.columns = ['HR', 'expectation']
    if ID is not None:
        predictions = pd.concat([ID, hr, expectation], axis=1)
        predictions.columns = ['ID', 'HR', 'expectation']
    else:
        predictions = pd.concat([hr, expectation], axis=1)
        predictions.columns = ['HR', 'expectation']
    return predictions

os.makedirs('results', exist_ok=True)
info = []
for subset, test_data in val_datasets.items():
    if subset in get_param_in_cwd('subsets'):
        results = get_prediction(cph, test_data, ID=test_data['ID'])
        results.to_csv(f'results/{task_type}cox_predictions_{subset}.csv', index=False)
        results['group'] = subset
        info.append(results)
        pd.merge(results, label_data[['ID', event_col, duration_col]], on='ID', how='inner').to_csv(f'features/{task_type}4xtile_{subset}.txt', 
                                                                                                    index=False, sep='\t')
info = pd.concat(info, axis=0)
info